# FoodEx2 Classification — Embedding & Product Matching

This notebook:
1. **Embeds** all 31 382 FoodEx2 terms using `paraphrase-multilingual-mpnet-base-v2`
2. **Loads** pre-computed product embeddings (or re-encodes from scratch)
3. **Matches** each product to its top-K FoodEx2 categories via cosine similarity (FAISS)
4. **Analyses** match quality (score distribution, coverage, most-matched terms)
5. **Exports** results for downstream use

> **Colab tips**
> - Runtime → Change runtime type → **T4 GPU** (encoding 31 K terms takes ~25 s on GPU, ~5 min on CPU)
> - Mount your Google Drive in Section 0 — all heavy artefacts (`.npy`) are cached there so they survive session resets

---
## 0. Colab Setup

In [ ]:
# Install dependencies (safe to re-run)
!pip install -q sentence-transformers faiss-cpu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── CONFIG ─────────────────────────────────────────────────────────────────
# Adjust DRIVE_BASE to the folder in your Drive that contains the project data.
CONFIG = dict(
    DRIVE_BASE        = '/content/drive/MyDrive/TFM',   # <-- change if needed
    MODEL_NAME        = 'paraphrase-multilingual-mpnet-base-v2',
    FOODEX2_CSV       = 'data/foodex2_nlp.csv',
    PRODUCTS_CSV      = 'data/products_nlp_full_sample_5pct.csv',
    # Pre-computed product embeddings from the clustering notebook
    # Set to None to always re-encode products.
    PRODUCT_EMB_NPY   = 'embeddings/product_embeddings.npy',
    # Where to save FoodEx2 embeddings
    FOODEX2_EMB_NPY   = 'embeddings/foodex2_embeddings.npy',
    FOODEX2_IDX_NPY   = 'embeddings/foodex2_term_codes.npy',
    # Matching
    TOP_K             = 5,
    THRESHOLD         = 0.3,    # cosine score below this = low confidence
    BATCH_SIZE        = 256,
    # Set True to ignore cached product .npy and re-encode from CSV
    RECOMPUTE_PRODUCTS = False,
)

# Build absolute paths
BASE = CONFIG['DRIVE_BASE']
for key in ('FOODEX2_CSV', 'PRODUCTS_CSV', 'PRODUCT_EMB_NPY',
            'FOODEX2_EMB_NPY', 'FOODEX2_IDX_NPY'):
    CONFIG[key] = os.path.join(BASE, CONFIG[key])

# Ensure embedding output directory exists
os.makedirs(os.path.dirname(CONFIG['FOODEX2_EMB_NPY']), exist_ok=True)

print('Config loaded:')
for k, v in CONFIG.items():
    print(f'  {k:25s} = {v}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import torch
import faiss
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

matplotlib.rcParams['figure.dpi'] = 120
pd.set_option('display.max_colwidth', 100)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 1. Load FoodEx2 Data

In [ ]:
foodex = pd.read_csv(CONFIG['FOODEX2_CSV'])
print(f'Shape: {foodex.shape}')
print(f'Columns: {foodex.columns.tolist()}')
foodex.head(3)

In [ ]:
# --- Coverage check ---
print('nlp_text null count:', foodex['nlp_text'].isna().sum())
print('nlp_text length stats:')
print(foodex['nlp_text'].str.len().describe().round(1))

# Flag very short texts (likely low-quality terms)
short_mask = foodex['nlp_text'].str.len() < 20
print(f'\nTerms with nlp_text < 20 chars: {short_mask.sum()}')
if short_mask.any():
    display(foodex[short_mask][['termCode', 'termExtendedName', 'nlp_text']].head(10))

In [ ]:
# Fill the rare nulls (13 terms) with the extended name as fallback
foodex['nlp_text'] = foodex['nlp_text'].fillna(foodex['termExtendedName'])
assert foodex['nlp_text'].isna().sum() == 0

# Build lookup arrays that will be reused by FAISS
foodex2_codes = foodex['termCode'].values           # shape (31382,)
foodex2_names = foodex['termExtendedName'].values   # shape (31382,)
foodex2_texts = foodex['nlp_text'].tolist()         # list of strings
print(f'Ready: {len(foodex2_texts):,} FoodEx2 terms')

---
## 2. Embed FoodEx2 Terms

We use the **same model** as the product clustering notebook so both embedding spaces are aligned.  
Embeddings are **cached to Drive** — re-running this cell skips encoding if the file already exists.

In [ ]:
def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    """Row-wise L2 normalisation.  Required so that IndexFlatIP == cosine similarity."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)   # avoid div-by-zero for zero vectors
    return (matrix / norms).astype(np.float32)

In [ ]:
emb_path   = CONFIG['FOODEX2_EMB_NPY']
codes_path = CONFIG['FOODEX2_IDX_NPY']

if os.path.exists(emb_path) and os.path.exists(codes_path):
    print('Loading cached FoodEx2 embeddings from Drive...')
    foodex2_emb   = np.load(emb_path)
    cached_codes  = np.load(codes_path, allow_pickle=True)
    # Verify the cached codes match the current CSV order
    if not np.array_equal(cached_codes, foodex2_codes):
        print('WARNING: cached term codes differ from CSV — re-encoding.')
        os.remove(emb_path)
        os.remove(codes_path)
    else:
        print(f'Loaded: {foodex2_emb.shape}')

if not os.path.exists(emb_path):
    print(f'Encoding {len(foodex2_texts):,} FoodEx2 terms on {DEVICE}...')
    model = SentenceTransformer(CONFIG['MODEL_NAME'], device=DEVICE)
    foodex2_emb = model.encode(
        foodex2_texts,
        batch_size=CONFIG['BATCH_SIZE'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,   # we normalise manually below
    )
    foodex2_emb = l2_normalize(foodex2_emb)
    np.save(emb_path,   foodex2_emb)
    np.save(codes_path, foodex2_codes)
    print(f'Saved to {emb_path}')
    print(f'Shape: {foodex2_emb.shape}')

# Sanity-check normalisation
norms = np.linalg.norm(foodex2_emb, axis=1)
print(f'Norm check — min: {norms.min():.6f}, max: {norms.max():.6f}  (should be ≈ 1.0)')

---
## 3. Load Product Embeddings

By default (`RECOMPUTE_PRODUCTS = False`) we load the `.npy` file saved by the clustering notebook.  
Set `RECOMPUTE_PRODUCTS = True` in CONFIG to re-encode from the CSV.

In [ ]:
products = pd.read_csv(CONFIG['PRODUCTS_CSV'])
print(f'Products shape: {products.shape}')
print(f'Columns: {products.columns.tolist()}')
products.head(2)

In [ ]:
prod_emb_path = CONFIG['PRODUCT_EMB_NPY']

if not CONFIG['RECOMPUTE_PRODUCTS'] and os.path.exists(prod_emb_path):
    print('Loading cached product embeddings from Drive...')
    product_emb = np.load(prod_emb_path).astype(np.float32)
    print(f'Loaded: {product_emb.shape}')

    if product_emb.shape[0] != len(products):
        print(f'WARNING: embedding rows ({product_emb.shape[0]}) != CSV rows ({len(products)})')
        print('Re-encoding products...')
        CONFIG['RECOMPUTE_PRODUCTS'] = True

if CONFIG['RECOMPUTE_PRODUCTS'] or not os.path.exists(prod_emb_path):
    print(f'Encoding {len(products):,} products on {DEVICE}...')
    if 'model' not in dir():
        model = SentenceTransformer(CONFIG['MODEL_NAME'], device=DEVICE)
    product_texts = products['nlp_text'].fillna('').tolist()
    product_emb = model.encode(
        product_texts,
        batch_size=CONFIG['BATCH_SIZE'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )
    product_emb = l2_normalize(product_emb)
    np.save(prod_emb_path, product_emb)
    print(f'Saved to {prod_emb_path}')

product_emb = l2_normalize(product_emb)   # ensure normalised regardless of source

norms = np.linalg.norm(product_emb, axis=1)
print(f'Norm check — min: {norms.min():.6f}, max: {norms.max():.6f}  (should be ≈ 1.0)')
print(f'Embedding dim match: products={product_emb.shape[1]}, foodex2={foodex2_emb.shape[1]}')
assert product_emb.shape[1] == foodex2_emb.shape[1], 'Dimension mismatch!'

---
## 4. Build FAISS Index for FoodEx2

`IndexFlatIP` (inner product) on L2-normalised vectors is equivalent to cosine similarity.  
All 31 382 FoodEx2 embeddings are loaded into the index — the search is exact (no approximation).

In [ ]:
dim = foodex2_emb.shape[1]   # 384
index = faiss.IndexFlatIP(dim)
index.add(foodex2_emb)

print(f'FAISS index built')
print(f'  Vectors indexed: {index.ntotal:,}')
print(f'  Embedding dim:   {dim}')

---
## 5. Matching — Top-K FoodEx2 per Product

In [ ]:
K     = CONFIG['TOP_K']
TRESH = CONFIG['THRESHOLD']
N     = len(products)

# FAISS search returns (scores, indices) of shape (N, K)
# We search in one shot — 57 K × 31 K fits in GPU/CPU memory comfortably
print(f'Searching top-{K} FoodEx2 matches for {N:,} products...')
scores, indices = index.search(product_emb, K)
print('Done.')

# scores  shape: (N, K) — cosine similarities (higher = better)
# indices shape: (N, K) — row indices into foodex2_emb / foodex2_codes
print(f'scores  shape: {scores.shape}')
print(f'indices shape: {indices.shape}')

In [ ]:
# --- Build long-format results DataFrame ---
records = []
for prod_idx in tqdm(range(N), desc='Building results'):
    prod_row = products.iloc[prod_idx]
    for rank in range(K):
        food_idx = int(indices[prod_idx, rank])
        score    = float(scores[prod_idx, rank])
        records.append({
            'product_idx'    : prod_idx,
            'product_id'     : prod_row.get('code', prod_row.get('id', prod_idx)),
            'product_name'   : str(prod_row.get('product_name_clean', prod_row.get('termExtendedName', '')))[:80],
            'match_rank'     : rank + 1,
            'foodex2_code'   : foodex2_codes[food_idx],
            'foodex2_name'   : foodex2_names[food_idx],
            'cosine_score'   : round(score, 4),
            'above_threshold': score >= TRESH,
        })

results_long = pd.DataFrame(records)
print(f'Long-format results: {results_long.shape}')
results_long.head(10)

In [ ]:
# --- Wide-format: one row per product, top-1 through top-K columns ---
wide_records = []
for prod_idx in range(N):
    prod_row = products.iloc[prod_idx]
    row = {
        'product_idx' : prod_idx,
        'product_id'  : prod_row.get('code', prod_row.get('id', prod_idx)),
        'product_name': str(prod_row.get('product_name_clean', ''))[:80],
    }
    for rank in range(K):
        food_idx = int(indices[prod_idx, rank])
        score    = float(scores[prod_idx, rank])
        row[f'top{rank+1}_code' ] = foodex2_codes[food_idx]
        row[f'top{rank+1}_name' ] = foodex2_names[food_idx]
        row[f'top{rank+1}_score'] = round(score, 4)
    wide_records.append(row)

results_wide = pd.DataFrame(wide_records)
print(f'Wide-format results: {results_wide.shape}')
results_wide.head(3)

---
## 6. Analysis

In [ ]:
# --- Top-1 score distribution ---
top1_scores = scores[:, 0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(top1_scores, bins=80, color='steelblue', edgecolor='white')
for t in (0.2, 0.3, 0.4):
    ax.axvline(t, color='tomato', linestyle='--', linewidth=1, label=f'threshold={t}')
ax.set_title('Top-1 FoodEx2 cosine similarity — product distribution')
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('Product count')
ax.legend()
plt.tight_layout()
plt.show()

print('Top-1 score stats:')
print(pd.Series(top1_scores).describe().round(4))

In [ ]:
# --- Coverage at different thresholds ---
print('Coverage (top-1 score >= threshold):')
for t in (0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5):
    pct = (top1_scores >= t).mean() * 100
    bar = '█' * int(pct / 2)
    print(f'  >= {t:.2f}: {pct:5.1f}%  {bar}')

print(f'\nUsing threshold = {TRESH}:')
above = (top1_scores >= TRESH).sum()
print(f'  Products with confident match: {above:,} / {N:,} ({above/N*100:.1f}%)')
print(f'  Low-confidence:                {N-above:,} ({(N-above)/N*100:.1f}%)')

In [ ]:
# --- Top 25 most-matched FoodEx2 terms (by top-1 assignment) ---
top1_df = results_long[results_long['match_rank'] == 1].copy()
top_terms = (
    top1_df.groupby(['foodex2_code', 'foodex2_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(25)
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_terms['foodex2_name'].str[:55], top_terms['count'], color='seagreen', edgecolor='white')
ax.invert_yaxis()
ax.set_title('Top 25 most-assigned FoodEx2 categories (top-1 matches)')
ax.set_xlabel('Product count')
plt.tight_layout()
plt.show()

display(top_terms.reset_index(drop=True))

In [ ]:
# --- Low-confidence products: spot-check ---
low_conf = top1_df[~top1_df['above_threshold']].copy()
print(f'Low-confidence products (top-1 < {TRESH}): {len(low_conf):,}')
print()
# Sample 15 for manual inspection
sample = low_conf.sample(min(15, len(low_conf)), random_state=42)[
    ['product_name', 'foodex2_name', 'cosine_score']
].sort_values('cosine_score')
display(sample)

In [ ]:
# --- High-confidence spot-check: manual validation ---
high_conf = top1_df[top1_df['cosine_score'] >= 0.5].sample(min(15, len(top1_df[top1_df['cosine_score'] >= 0.5])), random_state=42)
print(f'High-confidence sample (score >= 0.5):')
display(high_conf[['product_name', 'foodex2_name', 'cosine_score']].sort_values('cosine_score', ascending=False))

In [ ]:
# --- Score distribution per rank (top-1 vs top-2 vs top-3) ---
fig, ax = plt.subplots(figsize=(10, 4))
colours = ['steelblue', 'seagreen', 'coral', 'mediumpurple', 'goldenrod']
for rank in range(min(3, K)):
    ax.hist(scores[:, rank], bins=60, alpha=0.6, color=colours[rank],
            edgecolor='white', label=f'Top-{rank+1}')
ax.axvline(TRESH, color='red', linestyle='--', linewidth=1, label=f'threshold={TRESH}')
ax.set_title('Score distribution by match rank')
ax.set_xlabel('Cosine similarity')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Score gap between top-1 and top-2 (disambiguation confidence) ---
gap = scores[:, 0] - scores[:, 1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(gap, bins=60, color='mediumpurple', edgecolor='white')
ax.set_title('Score gap: top-1 minus top-2  (larger = more unambiguous match)')
ax.set_xlabel('Score gap')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(f'Median gap: {np.median(gap):.4f}')
print(f'Products with gap < 0.02 (ambiguous): {(gap < 0.02).sum():,}  ({(gap < 0.02).mean()*100:.1f}%)')

---
## 7. Export Results

In [ ]:
out_dir = os.path.join(BASE, 'results')
os.makedirs(out_dir, exist_ok=True)

wide_path = os.path.join(out_dir, 'products_foodex2_matches_top5.csv')
long_path = os.path.join(out_dir, 'products_foodex2_matches_long.csv')

results_wide.to_csv(wide_path, index=False)
results_long.to_csv(long_path, index=False)

print(f'Wide format saved → {wide_path}')
print(f'  Shape: {results_wide.shape}')
print(f'Long format saved → {long_path}')
print(f'  Shape: {results_long.shape}')

In [ ]:
# --- Final summary ---
print('=' * 55)
print('MATCHING SUMMARY')
print('=' * 55)
print(f'Model:                {CONFIG["MODEL_NAME"]}')
print(f'FoodEx2 terms indexed:{foodex2_emb.shape[0]:>10,}')
print(f'Products matched:     {product_emb.shape[0]:>10,}')
print(f'Top-K:                {K}')
print(f'Confidence threshold: {TRESH}')
print()
print('Top-1 cosine score:')
print(f'  mean  = {top1_scores.mean():.4f}')
print(f'  median= {np.median(top1_scores):.4f}')
print(f'  std   = {top1_scores.std():.4f}')
print()
for t in (0.2, 0.3, 0.4):
    pct = (top1_scores >= t).mean() * 100
    print(f'  >= {t}: {pct:.1f}% of products')
print()
print(f'Wide CSV:  {wide_path}')
print(f'Long CSV:  {long_path}')